In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(".")
print(" Ready")

 Ready


In [2]:
%%writefile agents/summarization_agent.py
from collections import defaultdict
from typing import Dict, List

from tools.llm_clients import call_llm
from tools.logger import log_agent_event

AGENT_NAME = "summarization_agent"


def _group_by_sub_question(fact_check_results: List[Dict]) -> Dict[str, List[Dict]]:
    grouped = defaultdict(list)
    for fact in fact_check_results:
        key = fact.get("sub_question", "unknown")
        grouped[key].append(fact)
    return dict(grouped)


def _build_summary_prompt(sub_question: str, facts: List[Dict]) -> str:
    confirmed = [f for f in facts if f["status"] == "CONFIRMED"]
    disputed = [f for f in facts if f["status"] == "DISPUTED"]
    unverified = [f for f in facts if f["status"] == "UNVERIFIED"]

    def fmt(fs):
        return "\n".join(f'- "{f["claim"]}" (reasoning: {f.get("reasoning", "")})' for f in fs) or "(none)"

    return f"""Write a short, honest research brief (3-6 sentences) answering: "{sub_question}"

You MUST follow these rules exactly:
- State CONFIRMED facts plainly, as established.
- For DISPUTED facts, explicitly say they are disputed and briefly describe BOTH sides —
  never silently pick one side or omit the disagreement.
- For UNVERIFIED facts, hedge them explicitly (e.g. "according to a single source...").
- Do not invent facts not listed below. Do not soften or hide disputes to make the
  summary sound cleaner.

CONFIRMED facts:
{fmt(confirmed)}

DISPUTED facts:
{fmt(disputed)}

UNVERIFIED facts:
{fmt(unverified)}

Write only the brief itself, no preamble.
"""


def summarize_sub_topic(sub_question: str, facts: List[Dict]) -> Dict:
    """Summarizes one sub-topic's fact-checked claims. Never raises: on failure,
    returns a summary that explicitly states it could not be generated, rather
    than silently omitting the sub-topic or fabricating a summary."""
    if not facts:
        return {
            "sub_question": sub_question,
            "summary": "No verified information available for this sub-topic.",
            "status": "no_facts",
            "fact_counts": {"CONFIRMED": 0, "DISPUTED": 0, "UNVERIFIED": 0},
        }

    counts = {"CONFIRMED": 0, "DISPUTED": 0, "UNVERIFIED": 0}
    for f in facts:
        counts[f["status"]] = counts.get(f["status"], 0) + 1

    try:
        summary_text = call_llm(_build_summary_prompt(sub_question, facts)).strip()
        status = "ok"
    except Exception as e:
        summary_text = (f"Summary could not be generated due to an error "
                        f"({type(e).__name__}). Raw fact counts: {counts}.")
        status = "failed"
        log_agent_event(AGENT_NAME, "summary_failed", sub_question=sub_question,
                        error=f"{type(e).__name__}: {e}")

    log_agent_event(AGENT_NAME, "summary_done", sub_question=sub_question, status=status,
                    **counts)

    return {
        "sub_question": sub_question,
        "summary": summary_text,
        "status": status,
        "fact_counts": counts,
    }


def summarization_node(state: Dict) -> Dict:
    """LangGraph node: summarizes every sub-topic's fact-checked claims."""
    fact_check_results = state.get("fact_check_results", [])
    if not fact_check_results:
        return {"errors": [{"agent": AGENT_NAME, "error": "no fact_check_results to summarize"}]}

    grouped = _group_by_sub_question(fact_check_results)
    summaries = [summarize_sub_topic(q, facts) for q, facts in grouped.items()]

    return {
        "summaries": summaries,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq",
                            "sub_topics": len(summaries)}],
    }

Writing agents/summarization_agent.py


In [3]:
from agents.web_search_agent import search_one
from agents.data_extraction_agent import extract_facts_from_result
from agents.fact_checking_agent import fact_check_facts
from agents.summarization_agent import summarize_sub_topic

search_result = search_one("What are the main safety risks of solid-state EV batteries?")
facts = extract_facts_from_result(search_result)
checked = fact_check_facts(facts)

summary = summarize_sub_topic(search_result["sub_question"], checked)
print("Status:", summary["status"])
print("Fact counts:", summary["fact_counts"])
print("\n--- Summary ---")
print(summary["summary"])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


 Gemini unavailable (ClientError), falling back to Tavily...


2026-10-03 19:47:50,451 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3
2026-10-03 19:47:51,598 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the main safety risks of solid-state EV batteries? facts_found=14
2026-10-03 19:47:52,823 | INFO | research_system | [fact_checking_agent] batch_done CONFIRMED=13 UNVERIFIED=1
2026-10-03 19:47:53,267 | INFO | research_system | [summarization_agent] summary_done sub_question=What are the main safety risks of solid-state EV batteries? status=ok CONFIRMED=13 DISPUTED=0 UNVERIFIED=1


Status: ok
Fact counts: {'CONFIRMED': 13, 'DISPUTED': 0, 'UNVERIFIED': 1}

--- Summary ---
Solid-state batteries mitigate several critical safety risks associated with traditional liquid electrolytes, as the solid electrolyte is confirmed to be thermally stable and non-combustible. This design reduces the risk of thermal runaway—a chain reaction that can lead to fire, smoke, or explosions—and eliminates the specific hazards of liquid leakage and dendrite formation. However, the technology is not immune to all safety concerns; according to a single source, solid-state batteries can still short circuit. Additionally, while the solid electrolyte improves thermal stability, the battery remains subject to general degradation, where continuous cycling and frequent rapid charging lead to a gradual decline in capacity.


In [4]:
from agents.summarization_agent import summarize_sub_topic

disputed_facts = [
    {"claim": "Solid-state batteries have a lower fire risk than liquid-electrolyte batteries.",
     "status": "DISPUTED", "reasoning": "contradicts the next claim", "sources": []},
    {"claim": "Solid-state batteries have the same fire risk as liquid-electrolyte batteries.",
     "status": "DISPUTED", "reasoning": "contradicts the previous claim", "sources": []},
    {"claim": "Tesla's headquarters is in Austin, Texas.",
     "status": "UNVERIFIED", "reasoning": "single source", "sources": []},
]

result = summarize_sub_topic("What is the fire risk of solid-state batteries?", disputed_facts)
print(result["summary"])

summary_lower = result["summary"].lower()
has_dispute_language = any(w in summary_lower for w in ["dispute", "disagree", "conflict", "unclear", "mixed"])
print("\n Summary acknowledges the dispute" if has_dispute_language else "\n❌ FAIL: summary silently smoothed over the dispute")

2026-10-03 19:56:44,381 | INFO | research_system | [summarization_agent] summary_done sub_question=What is the fire risk of solid-state batteries? status=ok CONFIRMED=0 DISPUTED=2 UNVERIFIED=1


There are no confirmed facts available regarding the specific fire risk of solid-state batteries. The relative safety profile is currently disputed, with one side arguing that solid-state batteries have a lower fire risk than liquid-electrolyte batteries, while the other side contends they have the same fire risk. No other verified information is provided to resolve this disagreement.

 Summary acknowledges the dispute


In [5]:
empty_result = summarize_sub_topic("A sub-topic with no facts", [])
print(empty_result)
assert empty_result["status"] == "no_facts"
assert "No verified information" in empty_result["summary"]
print("No-facts case handled correctly, no hallucinated content")

{'sub_question': 'A sub-topic with no facts', 'summary': 'No verified information available for this sub-topic.', 'status': 'no_facts', 'fact_counts': {'CONFIRMED': 0, 'DISPUTED': 0, 'UNVERIFIED': 0}}
No-facts case handled correctly, no hallucinated content


In [6]:
from agents.summarization_agent import summarization_node

state = {
    "fact_check_results": checked + disputed_facts  
}

for f in disputed_facts:
    f["sub_question"] = "What is the fire risk of solid-state batteries?"

out = summarization_node(state)
print(f"Generated {len(out['summaries'])} sub-topic summaries\n")
for s in out["summaries"]:
    print(f"[{s['status']}] {s['sub_question']}")
    print(f"  counts: {s['fact_counts']}")
    print(f"  {s['summary'][:150]}...\n")

2026-10-03 19:59:15,963 | INFO | research_system | [summarization_agent] summary_done sub_question=What are the main safety risks of solid-state EV batteries? status=ok CONFIRMED=13 DISPUTED=0 UNVERIFIED=1
2026-10-03 19:59:16,195 | INFO | research_system | [summarization_agent] summary_done sub_question=What is the fire risk of solid-state batteries? status=ok CONFIRMED=0 DISPUTED=2 UNVERIFIED=1


Generated 2 sub-topic summaries

[ok] What are the main safety risks of solid-state EV batteries?
  counts: {'CONFIRMED': 13, 'DISPUTED': 0, 'UNVERIFIED': 1}
  Solid-state batteries mitigate several critical safety risks associated with traditional liquid electrolytes, as the solid electrolyte is confirmed to...

[ok] What is the fire risk of solid-state batteries?
  counts: {'CONFIRMED': 0, 'DISPUTED': 2, 'UNVERIFIED': 1}
  There are no confirmed facts available regarding the specific fire risk of solid-state batteries. The relative safety profile is currently disputed, w...



In [ ]:
%%writefile tests/test_summarization_agent.py
import agents.summarization_agent as sa


def test_summarize_empty_facts_no_llm_call(monkeypatch):
    called = {"value": False}
    def fake_call(p):
        called["value"] = True
        return "should not be called"
    monkeypatch.setattr(sa, "call_llm", fake_call)
    result = sa.summarize_sub_topic("q", [])
    assert result["status"] == "no_facts"
    assert called["value"] is False


def test_summarize_llm_failure_does_not_crash(monkeypatch):
    def boom(p):
        raise RuntimeError("down")
    monkeypatch.setattr(sa, "call_llm", boom)
    facts = [{"claim": "A", "status": "CONFIRMED", "reasoning": "", "sources": []}]
    result = sa.summarize_sub_topic("q", facts)
    assert result["status"] == "failed"
    assert "error" in result["summary"].lower() or "RuntimeError" in result["summary"]


def test_summarize_counts_statuses_correctly(monkeypatch):
    monkeypatch.setattr(sa, "call_llm", lambda p: "a summary")
    facts = [
        {"claim": "A", "status": "CONFIRMED", "reasoning": "", "sources": []},
        {"claim": "B", "status": "CONFIRMED", "reasoning": "", "sources": []},
        {"claim": "C", "status": "DISPUTED", "reasoning": "", "sources": []},
        {"claim": "D", "status": "UNVERIFIED", "reasoning": "", "sources": []},
    ]
    result = sa.summarize_sub_topic("q", facts)
    assert result["fact_counts"] == {"CONFIRMED": 2, "DISPUTED": 1, "UNVERIFIED": 1}


def test_group_by_sub_question():
    facts = [
        {"claim": "A", "sub_question": "q1", "status": "CONFIRMED"},
        {"claim": "B", "sub_question": "q1", "status": "CONFIRMED"},
        {"claim": "C", "sub_question": "q2", "status": "DISPUTED"},
    ]
    grouped = sa._group_by_sub_question(facts)
    assert len(grouped) == 2
    assert len(grouped["q1"]) == 2
    assert len(grouped["q2"]) == 1


def test_node_aggregates_multiple_sub_topics(monkeypatch):
    monkeypatch.setattr(sa, "call_llm", lambda p: "a summary")
    state = {"fact_check_results": [
        {"claim": "A", "sub_question": "q1", "status": "CONFIRMED", "reasoning": "", "sources": []},
        {"claim": "B", "sub_question": "q2", "status": "CONFIRMED", "reasoning": "", "sources": []},
    ]}
    out = sa.summarization_node(state)
    assert len(out["summaries"]) == 2


def test_node_empty_fact_check_results():
    out = sa.summarization_node({"fact_check_results": []})
    assert out["errors"] and "summaries" not in out